In [1]:
import os
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import cv2
from tqdm import tqdm
import tensorflow as tf

In [2]:
base_project_folder = "./"

os.listdir(base_project_folder)

['augmented..csv',
 'augmented_df_3000_ground_truth.csv',
 'augmented_df_ground_truth.csv',
 'augmented_images',
 'augmented_images_3000',
 'augmented_new_vers',
 'backups_models',
 'ISIC2018_Task3_Test_GroundTruth.csv',
 'ISIC2018_Task3_Test_Input',
 'ISIC2018_Task3_Training_GroundTruth.csv',
 'ISIC2018_Task3_Training_Input',
 'ISIC2018_Task3_Validation_GroundTruth.csv',
 'ISIC2018_Task3_Validation_Input',
 'main.py',
 'myenv',
 'NN_training aug_also_top.ipynb',
 'NN_training.ipynb',
 'NN_training_with_rejector_mech.ipynb',
 'ordered_augmented_df.csv',
 'ordered_augmented_df_3000_unshuffeled.csv',
 'ordered_test_df.csv',
 'ordered_training_df.csv',
 'ordered_training_df_upper_limit_1500.csv',
 'ordered_training_df_upper_limit_3000.csv',
 'ordered_tranining_merged_3000.csv',
 'ordered_tranining_merged_3000.csv.zip',
 'ordered_validation_df.csv',
 'pixelated_augmented_3000.csv',
 'plain_model.h5',
 'plain_model.keras',
 'plain_model_best_perfoming.h5',
 'plain_model_with_augmentation_10

In [3]:
training_ground_truth = pd.read_csv(f"{base_project_folder}ISIC2018_Task3_Training_GroundTruth.csv")
validation_ground_truth = pd.read_csv(f"{base_project_folder}ISIC2018_Task3_Validation_GroundTruth.csv")
test_ground_truth = pd.read_csv(f"{base_project_folder}ISIC2018_Task3_Test_GroundTruth.csv")

training_ground_truth['image_path'] = training_ground_truth['image'].apply(lambda x: f"{base_project_folder}ISIC2018_Task3_Training_Input/{x}.jpg") 
validation_ground_truth['image_path'] = validation_ground_truth['image'].apply(lambda x: f"{base_project_folder}ISIC2018_Task3_Validation_Input/{x}.jpg") 
test_ground_truth['image_path'] = test_ground_truth['image'].apply(lambda x: f"{base_project_folder}ISIC2018_Task3_Test_Input/{x}.jpg")

print(training_ground_truth.head())

          image  MEL   NV  BCC  AKIEC  BKL   DF  VASC  \
0  ISIC_0024306  0.0  1.0  0.0    0.0  0.0  0.0   0.0   
1  ISIC_0024307  0.0  1.0  0.0    0.0  0.0  0.0   0.0   
2  ISIC_0024308  0.0  1.0  0.0    0.0  0.0  0.0   0.0   
3  ISIC_0024309  0.0  1.0  0.0    0.0  0.0  0.0   0.0   
4  ISIC_0024310  1.0  0.0  0.0    0.0  0.0  0.0   0.0   

                                         image_path  
0  ./ISIC2018_Task3_Training_Input/ISIC_0024306.jpg  
1  ./ISIC2018_Task3_Training_Input/ISIC_0024307.jpg  
2  ./ISIC2018_Task3_Training_Input/ISIC_0024308.jpg  
3  ./ISIC2018_Task3_Training_Input/ISIC_0024309.jpg  
4  ./ISIC2018_Task3_Training_Input/ISIC_0024310.jpg  


---

In [4]:
class_distribution = training_ground_truth.drop('image', axis=1).sum(axis=0)
print(class_distribution)

MEL                                                      1113.0
NV                                                       6705.0
BCC                                                       514.0
AKIEC                                                     327.0
BKL                                                      1099.0
DF                                                        115.0
VASC                                                      142.0
image_path    ./ISIC2018_Task3_Training_Input/ISIC_0024306.j...
dtype: object


In [5]:
for index, row in minority_class_df.iterrows():
    image_path = row['image']  # Assuming you have a column 'image' with image paths
    augmented_images = augment_image(image_path, num_augmentations=5)
    
    # Get the one-hot encoded label for this class
    one_hot_label = row.drop('image').values  # The rest of the columns are one-hot encoded labels
    
    # Create augmented rows with the same one-hot encoded label
    for augmented_image in augmented_images:
        augmented_data.append({
            'image': augmented_image,  # You may want to save the image or use its raw data
            **dict(zip(training_df.columns[1:], one_hot_label))  # Add the one-hot encoded label back
        })

augmented_df = pd.DataFrame(augmented_data)

# Combine the original and augmented DataFrame
balanced_df = pd.concat([training_df, augmented_df], ignore_index=True)

print(balanced_df.drop('image', axis=1).sum(axis=0))  # Check the new class distribution

NameError: name 'minority_class_df' is not defined

---

In [5]:
def preprocess_image_rgn(image_path, target_size=(224, 224)):
    try:
        image = tf.io.read_file(image_path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, target_size)
        image = image / 255.0
        image = tf.cast(image * 255, tf.uint8)
        return image.numpy()
    except Exception as e:
        print(f"Error processing {image_path}")
        print(e)
        return None


def build_dt_batch_processing(base_df, output_csv_path, batch_size=100, target_size=(224, 224)):
    
    pixel_columns = [f"pixel_{i}" for i in range(target_size[0] * target_size[1] * 3)]
    all_columns = list(base_df.columns) + pixel_columns


    # Ensure output file exists or create an empty one with headers
    if not os.path.exists(output_csv_path):
        pd.DataFrame(columns=all_columns).to_csv(output_csv_path, index=False)

    # Read existing rows from the output CSV to find processed images
    processed_images = set()
    if os.path.getsize(output_csv_path) > 0:  # Check if file is not empty
        processed_df = pd.read_csv(output_csv_path, usecols=["image"])  # Only load 'image' column
        processed_images = set(processed_df['image'].tolist())

    # Process in batches
    for i in range(0, len(base_df), batch_size):
        batch = base_df.iloc[i:i + batch_size]

        rows_to_add = []  # Store new rows to add to the CSV
        for idx, row in tqdm(batch.iterrows(), total=len(batch), desc=f"Processing batch {i // batch_size + 1}"):
            try:
                # Skip if the image is already processed
                if row['image'] in processed_images:
                    continue

                # Preprocess the image
                pixels = preprocess_image_rgn(row['image_path'], target_size=target_size)
                if pixels is None:
                    continue

                # Flatten pixels and add to row
                pixels = pixels.flatten()
                new_row = list(row) + list(pixels)
                rows_to_add.append(new_row)
            except Exception as e:
                print(f"Error processing image {row['image_path']} at index {idx}: {e}")

        # Append new rows to CSV file
        if rows_to_add:
            batch_df = pd.DataFrame(rows_to_add, columns=all_columns)
            batch_df.to_csv(output_csv_path, mode='a', header=False, index=False)
            print(f"Batch {i // batch_size + 1} saved to {output_csv_path}")

        # Update the set of processed images dynamically
        processed_images.update(batch['image'])

    print("Processing complete.")



In [ ]:
pixelated_augmented_input = os.path.join(base_project_folder, "augmented_df.csv")
augmented_df_ground_truth = pd.read_csv(augmented_df_ground_truth)
build_dt_batch_processing(augmented_df, output_csv_path=pixelated_augmented_input, batch_size=100)

augmented_df = pd.read_csv(pixelated_augmented_input)

----

In [5]:
pixelated_training_input = os.path.join(base_project_folder, "traning_df.csv")

In [ ]:
training_df = pd.read_csv(pixelated_training_input)

print(training_df.shape)
training_df.head()

In [ ]:
build_dt_batch_processing(training_ground_truth, output_csv_path=pixelated_training_input, batch_size=100)

training_df = pd.read_csv(pixelated_training_input)

print(training_df.shape)
training_df.head()

In [ ]:
chunk_size = 1000  #
for i,chunk in enumerate(pd.read_csv('./traning_df.csv', chunksize=chunk_size)):
    print(f"{i}.)")
    print(chunk.head())
    break

In [ ]:
df = pd.read_csv(pixelated_training_input)
pixel_cols = [col for col in df.columns if "pixel" in col]
other_cols = [col for col in df.columns if "pixel" not in col]

print(f"pixel col count: {len(pixel_cols)}")
print(f"other col count: {other_cols}")
df.shape

---

In [13]:
pixelated_validation_input = os.path.join(base_project_folder, "validation_df.csv")

In [ ]:
build_dt_batch_processing(validation_ground_truth, output_csv_path=pixelated_validation_input, batch_size=50)

validation_df = pd.read_csv(pixelated_validation_input)

print(validation_df.shape)
validation_df.head()

In [ ]:
pixelated_test_input = os.path.join(base_project_folder, "test_df.csv")

build_dt_batch_processing(test_ground_truth, output_csv_path=pixelated_test_input, batch_size=50)

test_df = pd.read_csv(pixelated_test_input)

print(test_df.shape)
test_df.head()

--------------

In [6]:
os.listdir("./")

['backups_models',
 'ISIC2018_Task3_Test_GroundTruth.csv',
 'ISIC2018_Task3_Test_Input',
 'ISIC2018_Task3_Training_GroundTruth.csv',
 'ISIC2018_Task3_Training_Input',
 'ISIC2018_Task3_Validation_GroundTruth.csv',
 'ISIC2018_Task3_Validation_Input',
 'main.py',
 'myenv',
 'NN_training.ipynb',
 'ordered_test_df.csv',
 'ordered_training_df.csv',
 'ordered_validation_df.csv',
 'plain_model.h5',
 'plain_model.keras',
 'plain_model_with_balancing.h5',
 'preparation.ipynb',
 'test_df.csv',
 'training_df.csv',
 'validation_df.csv']

In [56]:
# Define the function to rename the pixel columns
def rename_pixel_columns(columns):
    renamed_columns = []
    pixel_index = 0
    for col in columns:
        if col.startswith("pixel_"):
            # Determine the RGB channel (r, g, b)
            channel = ["r", "g", "b"][pixel_index % 3]
            renamed_columns.append(f"pixel_{pixel_index // 3}_{channel}")
            pixel_index += 1
        else:
            renamed_columns.append(col)
    return renamed_columns

In [15]:
# Input and output file paths
input_csv = "./training_df.csv"
output_csv = "./ordered_training_df.csv"

# Process the file to rename only the header
with open(input_csv, "r") as infile, open(output_csv, "w") as outfile:
    # Read the header (first line)
    header = infile.readline().strip().split(",")
    # Rename the columns
    new_header = rename_pixel_columns(header)
    # Write the new header to the output file
    outfile.write(",".join(new_header) + "\n")
    # Append the rest of the file line-by-line
    for line in infile:
        outfile.write(line)


In [12]:
input_csv = "./validation_df.csv"
output_csv = "./ordered_validation_df.csv"

# Process the file to rename only the header
with open(input_csv, "r") as infile, open(output_csv, "w") as outfile:
    # Read the header (first line)
    header = infile.readline().strip().split(",")
    # Rename the columns
    new_header = rename_pixel_columns(header)
    # Write the new header to the output file
    outfile.write(",".join(new_header) + "\n")
    # Append the rest of the file line-by-line
    for line in infile:
        outfile.write(line)


In [13]:
input_csv = "./test_df.csv"
output_csv = "./ordered_test_df.csv"

# Process the file to rename only the header
with open(input_csv, "r") as infile, open(output_csv, "w") as outfile:
    # Read the header (first line)
    header = infile.readline().strip().split(",")
    # Rename the columns
    new_header = rename_pixel_columns(header)
    # Write the new header to the output file
    outfile.write(",".join(new_header) + "\n")
    # Append the rest of the file line-by-line
    for line in infile:
        outfile.write(line)


----

In [4]:
header = []
with open(output_csv, "r") as infile:
    # Read the header (first line)
    header = infile.readline().strip().split(",")

In [ ]:
print(header[:15])
len(header)

------

In [1]:
import pandas as pd
from PIL import Image
from torchvision import transforms
import os
import shutil

In [6]:
base_project_folder = "./"

os.listdir(base_project_folder)

['augmented..csv',
 'augmented_df_3000_ground_truth.csv',
 'augmented_df_ground_truth.csv',
 'augmented_images',
 'augmented_images_3000',
 'augmented_new_vers',
 'backups_models',
 'ISIC2018_Task3_Test_GroundTruth.csv',
 'ISIC2018_Task3_Test_Input',
 'ISIC2018_Task3_Training_GroundTruth.csv',
 'ISIC2018_Task3_Training_Input',
 'ISIC2018_Task3_Validation_GroundTruth.csv',
 'ISIC2018_Task3_Validation_Input',
 'main.py',
 'myenv',
 'NN_training aug_also_top.ipynb',
 'NN_training.ipynb',
 'NN_training_with_rejector_mech.ipynb',
 'ordered_augmented_df.csv',
 'ordered_augmented_df_3000_unshuffeled.csv',
 'ordered_test_df.csv',
 'ordered_training_df.csv',
 'ordered_training_df_upper_limit_1500.csv',
 'ordered_training_df_upper_limit_3000.csv',
 'ordered_tranining_merged_3000.csv',
 'ordered_tranining_merged_3000.csv.zip',
 'ordered_validation_df.csv',
 'pixelated_augmented_3000.csv',
 'plain_model.h5',
 'plain_model.keras',
 'plain_model_best_perfoming.h5',
 'plain_model_with_augmentation_10

In [7]:
training_ground_truth = pd.read_csv(f"{base_project_folder}ISIC2018_Task3_Training_GroundTruth.csv")
training_ground_truth['image_path'] = training_ground_truth['image'].apply(lambda x: f"{base_project_folder}ISIC2018_Task3_Training_Input/{x}.jpg") 
print(training_ground_truth.head())

          image  MEL   NV  BCC  AKIEC  BKL   DF  VASC  \
0  ISIC_0024306  0.0  1.0  0.0    0.0  0.0  0.0   0.0   
1  ISIC_0024307  0.0  1.0  0.0    0.0  0.0  0.0   0.0   
2  ISIC_0024308  0.0  1.0  0.0    0.0  0.0  0.0   0.0   
3  ISIC_0024309  0.0  1.0  0.0    0.0  0.0  0.0   0.0   
4  ISIC_0024310  1.0  0.0  0.0    0.0  0.0  0.0   0.0   

                                         image_path  
0  ./ISIC2018_Task3_Training_Input/ISIC_0024306.jpg  
1  ./ISIC2018_Task3_Training_Input/ISIC_0024307.jpg  
2  ./ISIC2018_Task3_Training_Input/ISIC_0024308.jpg  
3  ./ISIC2018_Task3_Training_Input/ISIC_0024309.jpg  
4  ./ISIC2018_Task3_Training_Input/ISIC_0024310.jpg  


In [8]:
class_distribution = training_ground_truth.drop(['image', 'image_path'], axis=1).sum(axis=0)
print(class_distribution)

MEL      1113.0
NV       6705.0
BCC       514.0
AKIEC     327.0
BKL      1099.0
DF        115.0
VASC      142.0
dtype: float64


In [9]:
training_ground_truth.head()

,image,MEL,NV,BCC,AKIEC,BKL,DF,VASC,image_path
0,ISIC_0024306,0.0,1.0,0.0,0.0,0.0,0.0,0.0,./ISIC2018_Task3_Training_Input/ISIC_0024306.jpg
1,ISIC_0024307,0.0,1.0,0.0,0.0,0.0,0.0,0.0,./ISIC2018_Task3_Training_Input/ISIC_0024307.jpg
2,ISIC_0024308,0.0,1.0,0.0,0.0,0.0,0.0,0.0,./ISIC2018_Task3_Training_Input/ISIC_0024308.jpg
3,ISIC_0024309,0.0,1.0,0.0,0.0,0.0,0.0,0.0,./ISIC2018_Task3_Training_Input/ISIC_0024309.jpg
4,ISIC_0024310,1.0,0.0,0.0,0.0,0.0,0.0,0.0,./ISIC2018_Task3_Training_Input/ISIC_0024310.jpg


In [15]:
from torchvision import transforms

def augment_image(image_path, num_augmentations=5):
    image = Image.open(image_path).convert("RGB")
    augmented_images = []
    
    for _ in range(num_augmentations):
        augmented_images.append(augmentation_transform(image))
    
    return augmented_images



augmentation_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(128, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomAffine(degrees=10, shear=5),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1))
])

In [11]:
minority_classes = class_distribution[class_distribution == class_distribution.min()].index.tolist()
print("Minority classes:", minority_classes)

Minority classes: ['DF']


In [12]:
augmented_images_dir = './augmented_new_vers/'
if not os.path.exists(augmented_images_dir):
    os.makedirs(augmented_images_dir)

In [13]:
augmented_images_dir

'./augmented_new_vers/'

MEL      1113.0
NV       6705.0
BCC       514.0
AKIEC     327.0
BKL      1099.0
DF        115.0
VASC      142.0

In [ ]:
augmented_data = []

minor_class = [["VASC"], ["DF"], ["BCC"], ["AKIEC"], ["BCC"]]
num_augmentations = [9, 9, 2, 3, 2]

for classes_index, minority_classes in enumerate(minor_class):
    for index, row in training_ground_truth.iterrows():
        # If the row corresponds to one of the minority classes (has a 1 in any of the minority columns)``
        if row[minority_classes].sum() > 0:
            image_path = row['image_path']
            augmented_images = augment_image(image_path, num_augmentations=num_augmentations[classes_index])
            
            # Create new rows for the augmented images with the same labels
            for i, augmented_image in enumerate(augmented_images):
                new_image_name = f"{row['image']}_aug_{i}.jpg"
                augmented_image_path = os.path.join(augmented_images_dir, new_image_name)  # Use full path
                
                # Save augmented image (optional, if you want to save it as a new file)
                augmented_image.save(augmented_image_path)
                
                # Add new row with the augmented image
                augmented_row = row.copy()
                augmented_row['image_path'] = augmented_image_path
                augmented_row['image'] = new_image_name  # Or you can keep the original name if preferred
                
                augmented_data.append(augmented_row)

AttributeError: 'Image' object has no attribute 'shape'

In [31]:
augmented_df = pd.DataFrame(augmented_data)
balanced_df = pd.concat([training_ground_truth, augmented_df], ignore_index=True)

In [32]:
balanced_df

,image,MEL,NV,BCC,AKIEC,BKL,DF,VASC,image_path
0,ISIC_0024306,0.0,1.0,0.0,0.0,0.0,0.0,0.0,./ISIC2018_Task3_Training_Input/ISIC_0024306.jpg
1,ISIC_0024307,0.0,1.0,0.0,0.0,0.0,0.0,0.0,./ISIC2018_Task3_Training_Input/ISIC_0024307.jpg
2,ISIC_0024308,0.0,1.0,0.0,0.0,0.0,0.0,0.0,./ISIC2018_Task3_Training_Input/ISIC_0024308.jpg
3,ISIC_0024309,0.0,1.0,0.0,0.0,0.0,0.0,0.0,./ISIC2018_Task3_Training_Input/ISIC_0024309.jpg
4,ISIC_0024310,1.0,0.0,0.0,0.0,0.0,0.0,0.0,./ISIC2018_Task3_Training_Input/ISIC_0024310.jpg
...,...,...,...,...,...,...,...,...,...
15360,ISIC_0034276_aug_1.jpg,0.0,0.0,1.0,0.0,0.0,0.0,0.0,./augmented_images/ISIC_0034276_aug_1.jpg
15361,ISIC_0034299_aug_0.jpg,0.0,0.0,1.0,0.0,0.0,0.0,0.0,./augmented_images/ISIC_0034299_aug_0.jpg
15362,ISIC_0034299_aug_1.jpg,0.0,0.0,1.0,0.0,0.0,0.0,0.0,./augmented_images/ISIC_0034299_aug_1.jpg
15363,ISIC_0034306_aug_0.jpg,0.0,0.0,1.0,0.0,0.0,0.0,0.0,./augmented_images/ISIC_0034306_aug_0.jpg


In [33]:
augmented_df[augmented_df["image"].str.contains("ISIC_0034169")]

,image,MEL,NV,BCC,AKIEC,BKL,DF,VASC,image_path
9863,ISIC_0034169_aug_0.jpg,0.0,0.0,0.0,0.0,0.0,1.0,0.0,./augmented_images/ISIC_0034169_aug_0.jpg
9863,ISIC_0034169_aug_1.jpg,0.0,0.0,0.0,0.0,0.0,1.0,0.0,./augmented_images/ISIC_0034169_aug_1.jpg
9863,ISIC_0034169_aug_2.jpg,0.0,0.0,0.0,0.0,0.0,1.0,0.0,./augmented_images/ISIC_0034169_aug_2.jpg
9863,ISIC_0034169_aug_3.jpg,0.0,0.0,0.0,0.0,0.0,1.0,0.0,./augmented_images/ISIC_0034169_aug_3.jpg
9863,ISIC_0034169_aug_4.jpg,0.0,0.0,0.0,0.0,0.0,1.0,0.0,./augmented_images/ISIC_0034169_aug_4.jpg
9863,ISIC_0034169_aug_5.jpg,0.0,0.0,0.0,0.0,0.0,1.0,0.0,./augmented_images/ISIC_0034169_aug_5.jpg
9863,ISIC_0034169_aug_6.jpg,0.0,0.0,0.0,0.0,0.0,1.0,0.0,./augmented_images/ISIC_0034169_aug_6.jpg
9863,ISIC_0034169_aug_7.jpg,0.0,0.0,0.0,0.0,0.0,1.0,0.0,./augmented_images/ISIC_0034169_aug_7.jpg
9863,ISIC_0034169_aug_8.jpg,0.0,0.0,0.0,0.0,0.0,1.0,0.0,./augmented_images/ISIC_0034169_aug_8.jpg


In [34]:
training_ground_truth[training_ground_truth["image"].str.contains("ISIC_0034169")]

,image,MEL,NV,BCC,AKIEC,BKL,DF,VASC,image_path
9863,ISIC_0034169,0.0,0.0,0.0,0.0,0.0,1.0,0.0,./ISIC2018_Task3_Training_Input/ISIC_0034169.jpg


In [35]:
augmented_df.to_csv('./augmented_df_ground_truth.csv', index=False)

In [16]:

# Convert augmented data into a new DataFrame
augmented_df = pd.DataFrame(augmented_data)

# Combine original and augmented DataFrames
balanced_df = pd.concat([training_ground_truth, augmented_df], ignore_index=True)

# Check the new class distribution
print(balanced_df.drop(['image', 'image_path'], axis=1).sum(axis=0))

# Optionally, save the new balanced DataFrame and augmented images if desired
balanced_df.to_csv('balanced_training_ground_truth.csv', index=False)

# If you want to move augmented images to a specific directory:
# Ensure the 'augmented_images' folder exists
if not os.path.exists('augmented_images'):
    os.makedirs('augmented_images')

# Optional: Save augmented images
for augmented_image in augmented_data:
    image_path = augmented_image['image_path']
    new_image_path = f"./augmented_images/{augmented_image['image']}.jpg"
    shutil.copy(image_path, new_image_path)  # Save the image to the new folder

NameError: name 'augmented_data' is not defined

In [45]:
augmented_df.to_csv('./augmented_df_ground_truth.csv', index=False)

In [ ]:
augmented_df.sum(axis=0)

In [ ]:
balanced_df.sum(axis=0)

In [ ]:

minority_classes_VASC = ["VASC"]
for index, row in training_ground_truth.iterrows():
    # If the row corresponds to one of the minority classes (has a 1 in any of the minority columns)
    if row[minority_classes].sum() > 0:
        image_path = row['image_path']
        augmented_images = augment_image(image_path, num_augmentations=5)
        
        # Create new rows for the augmented images with the same labels
        for i, augmented_image in enumerate(augmented_images):
            new_image_name = f"{row['image']}_aug_{i}.jpg"
            augmented_image_path = os.path.join(augmented_images_dir, new_image_name)  # Use full path
            
            # Save augmented image (optional, if you want to save it as a new file)
            augmented_image.save(augmented_image_path)
            
            # Add new row with the augmented image
            augmented_row = row.copy()
            augmented_row['image_path'] = augmented_image_path
            augmented_row['image'] = new_image_name  # Or you can keep the original name if preferred
            
            augmented_data.append(augmented_row)

----

In [ ]:
class_distribution = training_ground_truth.drop(['image', 'image_path'], axis=1).sum(axis=0)
majority_class_size = class_distribution.max()
print(f'Majority class size: {majority_class_size}')

# List the classes
classes = class_distribution.index.tolist()

augmented_data = []

# Augment images for the minority classes to balance dataset
for class_label in classes:
    # Get the rows corresponding to the current class
    class_df = training_ground_truth[training_ground_truth[class_label] == 1]

    # Calculate how many samples we need to generate for this class
    num_augmentations_needed = majority_class_size - len(class_df)
    
    # If this class is underrepresented, apply augmentation
    if num_augmentations_needed > 0:
        print(f"Augmenting class {class_label} with {num_augmentations_needed} new samples.")
        
        for index, row in class_df.iterrows():
            image_path = row['image_path']
            augmented_images = augment_image(image_path, num_augmentations=int(num_augmentations_needed))
            
            # Create new rows for the augmented images with the same labels
            for i, augmented_image in enumerate(augmented_images):
                new_image_name = f"{row['image']}_aug_{i}.jpg"
                augmented_image_path = os.path.join('./augmented_images', new_image_name)  # Ensure dir exists
                
                # Save augmented image (optional, if you want to save it as a new file)
                augmented_image.save(augmented_image_path)
                
                # Add new row with the augmented image
                augmented_row = row.copy()
                augmented_row['image_path'] = augmented_image_path
                augmented_row['image'] = new_image_name  # Or keep the original name if preferred
                
                augmented_data.append(augmented_row)

# Convert augmented data into a new DataFrame
augmented_df = pd.DataFrame(augmented_data)



In [ ]:
# Combine original and augmented DataFrames
balanced_df = pd.concat([training_ground_truth, augmented_df], ignore_index=True)

# Check the new class distribution
balanced_class_distribution = balanced_df.drop(['image', 'image_path'], axis=1).sum(axis=0)
print("Balanced class distribution:", balanced_class_distribution)

# Save the balanced DataFrame
balanced_df.to_csv('balanced_training_ground_truth.csv', index=False)

-----

In [45]:
augmented_df.head()

,image,MEL,NV,BCC,AKIEC,BKL,DF,VASC,image_path
64,ISIC_0024370_aug_0.jpg,0.0,0.0,0.0,0.0,0.0,0.0,1.0,./augmented_images/ISIC_0024370_aug_0.jpg
64,ISIC_0024370_aug_1.jpg,0.0,0.0,0.0,0.0,0.0,0.0,1.0,./augmented_images/ISIC_0024370_aug_1.jpg
64,ISIC_0024370_aug_2.jpg,0.0,0.0,0.0,0.0,0.0,0.0,1.0,./augmented_images/ISIC_0024370_aug_2.jpg
64,ISIC_0024370_aug_3.jpg,0.0,0.0,0.0,0.0,0.0,0.0,1.0,./augmented_images/ISIC_0024370_aug_3.jpg
64,ISIC_0024370_aug_4.jpg,0.0,0.0,0.0,0.0,0.0,0.0,1.0,./augmented_images/ISIC_0024370_aug_4.jpg


In [ ]:
# pixelated_augmented_input = os.path.join(base_project_folder, "/augmented_df.csv")
# pixelated_augmented_df_input = "./augmented.csv"
pixelated_augmented_df_input =  "./augmented_new_vers.csv"
pixelated_augmented_input

'/augmented_df.csv'

In [53]:
def build_dt_batch_processing_a(base_df, output_csv_path, batch_size=100, target_size=(224, 224)):
    
    pixel_columns = [f"pixel_{i}" for i in range(target_size[0] * target_size[1] * 3)]
    all_columns = list(base_df.columns) + pixel_columns


    # Ensure output file exists or create an empty one with headers
    if not os.path.exists(output_csv_path):
        pd.DataFrame(columns=all_columns).to_csv(output_csv_path, index=False)

    # Read existing rows from the output CSV to find processed images
    processed_images = set()
    if os.path.getsize(output_csv_path) > 0:  # Check if file is not empty
        processed_df = pd.read_csv(output_csv_path, usecols=["image"])  # Only load 'image' column
        processed_images = set(processed_df['image'].tolist())

    # Process in batches
    for i in range(0, len(base_df), batch_size):
        batch = base_df.iloc[i:i + batch_size]

        rows_to_add = []  # Store new rows to add to the CSV
        for idx, row in tqdm(batch.iterrows(), total=len(batch), desc=f"Processing batch {i // batch_size + 1}"):
            try:
                # Skip if the image is already processed
                if row['image'] in processed_images:
                    continue

                # Preprocess the image
                pixels = preprocess_image_rgn(row['image_path'], target_size=target_size)
                if pixels is None:
                    continue

                # Flatten pixels and add to row
                pixels = pixels.flatten()
                new_row = list(row) + list(pixels)
                rows_to_add.append(new_row)
            except Exception as e:
                print(f"Error processing image {row['image_path']} at index {idx}: {e}")

        # Append new rows to CSV file
        if rows_to_add:
            batch_df = pd.DataFrame(rows_to_add, columns=all_columns)
            batch_df.to_csv(output_csv_path, mode='a', header=False, index=False)
            print(f"Batch {i // batch_size + 1} saved to {output_csv_path}")

        # Update the set of processed images dynamically
        processed_images.update(batch['image'])

    print("Processing complete.")


In [54]:

build_dt_batch_processing_a(augmented_df, output_csv_path=pixelated_augmented_df_input, batch_size=100)

augmented_df = pd.read_csv(pixelated_augmented_input)

Processing batch 1: 100%|██████████| 100/100 [00:00<00:00, 101.52it/s]


Batch 1 saved to ./augmented..csv


Processing batch 2: 100%|██████████| 100/100 [00:00<00:00, 106.61it/s]


Batch 2 saved to ./augmented..csv


Processing batch 3: 100%|██████████| 100/100 [00:00<00:00, 105.71it/s]


...

Batch 43 saved to ./augmented..csv


Processing batch 44: 100%|██████████| 100/100 [00:00<00:00, 487.81it/s]


Batch 44 saved to ./augmented..csv


Processing batch 54: 100%|██████████| 50/50 [00:00<00:00, 12494.20it/s]

Processing complete.


----

In [57]:
# Input and output file paths
input_csv = "./augmented..csv"
output_csv = "./ordered_augmented._df.csv"

# Process the file to rename only the header
with open(input_csv, "r") as infile, open(output_csv, "w") as outfile:
    # Read the header (first line)
    header = infile.readline().strip().split(",")
    # Rename the columns
    new_header = rename_pixel_columns(header)
    # Write the new header to the output file
    outfile.write(",".join(new_header) + "\n")
    # Append the rest of the file line-by-line
    for line in infile:
        outfile.write(line)


In [60]:
header = []
first_row = []
with open(output_csv, "r") as infile:
    # Read the header (first line)
    header = infile.readline().strip().split(",")
    first_row = infile.readline().strip().split(",")

In [61]:
header

['image',
 'MEL',
 'NV',
 'BCC',
 'AKIEC',
 'BKL',
 'DF',
 'VASC',
 'image_path',
 'pixel_0_r',
 'pixel_0_g',
 'pixel_0_b',
 'pixel_1_r',
 'pixel_1_g',
 'pixel_1_b',
 'pixel_2_r',
 'pixel_2_g',
 'pixel_2_b',
 'pixel_3_r',
 'pixel_3_g',
 'pixel_3_b',
 'pixel_4_r',
 'pixel_4_g',
 'pixel_4_b',
 'pixel_5_r',
 'pixel_5_g',
 'pixel_5_b',
 'pixel_6_r',
 'pixel_6_g',
 'pixel_6_b',
 'pixel_7_r',
 'pixel_7_g',
 'pixel_7_b',
 'pixel_8_r',
 'pixel_8_g',
 'pixel_8_b',
 'pixel_9_r',
 'pixel_9_g',
 'pixel_9_b',
 'pixel_10_r',
 'pixel_10_g',
 'pixel_10_b',
 'pixel_11_r',
 'pixel_11_g',
 'pixel_11_b',
 'pixel_12_r',
 'pixel_12_g',
 'pixel_12_b',
 'pixel_13_r',
 'pixel_13_g',
 'pixel_13_b',
 'pixel_14_r',
 'pixel_14_g',
 'pixel_14_b',
 'pixel_15_r',
 'pixel_15_g',
 'pixel_15_b',
 'pixel_16_r',
 'pixel_16_g',
 'pixel_16_b',
 'pixel_17_r',
 'pixel_17_g',
 'pixel_17_b',
 'pixel_18_r',
 'pixel_18_g',
 'pixel_18_b',
 'pixel_19_r',
 'pixel_19_g',
 'pixel_19_b',
 'pixel_20_r',
 'pixel_20_g',
 'pixel_20_b',
 'p

In [62]:
first_row

['ISIC_0024370_aug_0.jpg',
 '0.0',
 '0.0',
 '0.0',
 '0.0',
 '0.0',
 '0.0',
 '1.0',
 './augmented_images/ISIC_0024370_aug_0.jpg',
 '183',
 '127',
 '136',
 '183',
 '127',
 '136',
 '183',
 '127',
 '136',
 '183',
 '127',
 '136',
 '184',
 '128',
 '137',
 '184',
 '128',
 '137',
 '185',
 '129',
 '138',
 '185',
 '129',
 '138',
 '185',
 '129',
 '138',
 '185',
 '129',
 '138',
 '185',
 '129',
 '138',
 '185',
 '129',
 '138',
 '185',
 '129',
 '138',
 '184',
 '128',
 '137',
 '184',
 '128',
 '137',
 '184',
 '128',
 '137',
 '184',
 '128',
 '137',
 '184',
 '128',
 '137',
 '185',
 '129',
 '138',
 '185',
 '129',
 '138',
 '185',
 '129',
 '138',
 '185',
 '129',
 '138',
 '185',
 '129',
 '138',
 '185',
 '129',
 '138',
 '184',
 '128',
 '137',
 '184',
 '128',
 '137',
 '184',
 '128',
 '137',
 '183',
 '127',
 '137',
 '182',
 '126',
 '138',
 '182',
 '126',
 '139',
 '182',
 '126',
 '139',
 '182',
 '126',
 '139',
 '182',
 '126',
 '139',
 '182',
 '126',
 '139',
 '182',
 '126',
 '139',
 '182',
 '126',
 '139',
 '181',

----

In [8]:
#  undersampling fomr top (NV):

input_csv = "./ordered_training_df.csv"
output_csv_base  = "./ordered_training_df"
chunk_size = 1000  
limits = [1500, 3000]

reduced_nv_data = []

In [9]:

def process_and_save_nv_class(file_path, chunk_size=1000, limit= 1500):

    output_nv_file = output_csv_base + f"_upper_limit_{limit}.csv"
    chunk_iter = pd.read_csv(file_path, chunksize=chunk_size)
    first_chunk = True
    nv_samples_added = 0
    print(f"Processing with NV limit = {limit}")
    
    for i, df_chunk in enumerate(chunk_iter):
        print(f"Processing chunk {i}...")

        # Check if NV column exists
        if 'NV' not in df_chunk.columns:
            raise KeyError("'NV' column not found in dataset. Ensure labels are one-hot encoded.")

        df_nv = df_chunk[df_chunk['NV'] == 1]
        df_other_classes = df_chunk[df_chunk['NV'] != 1]  # K

        # Calculate how many NV samples are needed to reach the limit
        remaining_nv_samples = limit - nv_samples_added

        if remaining_nv_samples > 0:
            # Sample only the required amount of NV rows
            df_nv_reduced = df_nv.sample(n=min(len(df_nv), remaining_nv_samples), random_state=42)

            # Update the count of NV samples added
            nv_samples_added += len(df_nv_reduced)
        else:
            # If the limit is reached, no NV samples are added
            df_nv_reduced = pd.DataFrame()

        df_combined = pd.concat([df_nv_reduced, df_other_classes])
        df_combined.to_csv(output_nv_file, mode='a', header=first_chunk, index=False)
        first_chunk = False



for limit in limits:
    process_and_save_nv_class(input_csv, chunk_size, limit)

Processing with NV limit = 1500
Processing chunk 0...
Processing chunk 1...
Processing chunk 2...
Processing chunk 3...
Processing chunk 4...
Processing chunk 5...
Processing chunk 6...
Processing chunk 7...
Processing chunk 8...
Processing chunk 9...
Processing chunk 10...
Processing with NV limit = 3000
Processing chunk 0...
Processing chunk 1...
Processing chunk 2...
Processing chunk 3...
Processing chunk 4...
Processing chunk 5...
Processing chunk 6...
Processing chunk 7...
Processing chunk 8...
Processing chunk 9...
Processing chunk 10...


In [9]:
with open(f"{input_csv}", "r") as f:
    row_count = sum(1 for _ in f) - 1
print(row_count)

10015


In [10]:
limit = 1500
output_nv_file = output_csv_base + f"_upper_limit_{limit}.csv"

with open(f"{output_nv_file}", "r") as f:
    row_count = sum(1 for _ in f) - 1
print(row_count)

4810


In [11]:
limit = 3000
output_nv_file = output_csv_base + f"_upper_limit_{limit}.csv"

with open(f"{output_nv_file}", "r") as f:
    row_count = sum(1 for _ in f) - 1
print(row_count)

6310


In [12]:
limit= 1500
chunk_size = 3
output_nv_file = output_csv_base + f"_upper_limit_{limit}.csv"
chunk_iter = pd.read_csv(output_nv_file, chunksize=chunk_size)
first_chunk = True

for i, df_chunk in enumerate(chunk_iter):
    print(f"Processing chunk {i}...")
    print(df_chunk["NV"])
    break


Processing chunk 0...
0    1.0
1    1.0
2    1.0
Name: NV, dtype: float64


In [13]:
chunk_size = 3
chunk_iter = pd.read_csv(input_csv, chunksize=chunk_size)
first_chunk = True

for i, df_chunk in enumerate(chunk_iter):
    print(f"Processing chunk {i}...")
    print(df_chunk["NV"])
    break


Processing chunk 0...
0    1.0
1    1.0
2    1.0
Name: NV, dtype: float64
